# 04 · Memória em uma conversa

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from pprint import pprint
from src.config import DATA

<a id="coala"></a>

**CoALA.** Perfil e skills são memória semântica; vagas vistas e descartadas são
episódios; filtros que o usuário quer manter são regras procedurais. Uma janela
retém os últimos turnos; o histórico anterior é resumido. O índice vetorial de
sessão fica separado dos documentos para preservar a origem de cada dado.

| Ferramenta | Arquitetura | Uso típico |
|---|---|---|
| LangMem | Componentes de extração e manutenção sobre armazenamento do agente | Agentes integrados ao ecossistema LangGraph |
| Mem0 | Camada de extração e recuperação de memórias por escopo | Aplicações que precisam reutilizar preferências entre sessões |
| Letta | Estado e blocos de memória administrados pelo agente | Agentes de longa duração |

O FAISS local reduz dependências para a avaliação. O protótipo não persiste o
histórico conversacional entre reinícios do servidor.

In [2]:
from src.demo import buffer_demo, QUESTIONS
pprint(buffer_demo())

{'history': 'Human: Mostre vagas remotas de Python\n'
            'AI: Encontrei vagas de Python.\n'
            'Human: Só as que aceitam Brasil\n'
            'AI: Mantive o cargo Python e apliquei Brasil.'}


/Users/fernandotrinta/Desktop/Infnet/src/demo.py:125: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()
/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [3]:
from src.agente import Radar
from src.rules import interpretar
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temp:
    radar = Radar(db_path=Path(temp) / 'vagas.db')
    for question in QUESTIONS:
        response = radar.chat('sessao-1', question)
        print(question, '→', response.answer)
    pprint(interpretar(QUESTIONS[2]).model_dump(exclude_none=True))
    print('Sessão isolada:', radar.chat('sessao-2', QUESTIONS[2]).answer)
    session = radar.session('sessao-1')
    session.trim(limit=4)
    print('Janela:', len(session.history.messages), 'mensagens')
    print('Resumo:', session.summary[:300])
    pprint(session.retrieve('qual vaga descartada e por quê?'))

/var/folders/fp/5_zj8d4n2g1fr35b3rkfs9c40000gn/T/ipykernel_11455/1196064145.py:5: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  radar = Radar(db_path=Path(temp) / 'vagas.db')
/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1112 23:06:24.562000 11455 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 10888.69it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22590.91it/s]

Mostre vagas remotas de Python → 2 vaga(s) recomendada(s); 1 pendente(s); 2 descartada(s) na amostra exibida.
1. Senior Python Backend Engineer · Aurora Labs.
2. Python Platform Engineer · Orbit Works.
Só as que aceitam Brasil → 2 vaga(s) recomendada(s); 1 pendente(s); 2 descartada(s) na amostra exibida.
1. Senior Python Backend Engineer · Aurora Labs.
2. Python Platform Engineer · Orbit Works.
A lista numerada inclui apenas vagas com elegibilidade confirmada para o Brasil. Os demais casos aparecem como evidência.
Converta o salário da segunda → Python Platform Engineer: EUR 5,400.00 por mês ≈ R$ 32,400.00 por mês. Cotação fictícia de 2026-01-01 (1 EUR = R$ 6.00).
Qual combina mais com meu currículo? → Senior Python Backend Engineer é a melhor correspondência da lista atual. Habilidades em comum: Python, FastAPI, PostgreSQL.
{'entidades': {'posicao': 2}, 'intencao': 'cambio'}
Sessão isolada: Não encontrei essa posição na última lista desta sessão. Busque vagas primeiro.
Janela: 4 mensa

## Evidência

- **3.1** → [coala](#coala): Memória semântica, episódica e procedural.
- **3.2** → [turnos](#turnos): Janela, resumo e recuperação vetorial.
- **3.3** → [coala](#coala): Comparação de três ferramentas.
- **3.4** → [buffer](#buffer): ConversationBufferMemory em dois turnos.
- **3.5** → [turnos](#turnos): Intenção de câmbio e posição 2.
- **3.6** → [turnos](#turnos): Quatro turnos reais da API interna do agente, com sessão isolada.